<a href="https://colab.research.google.com/github/syedmahmoodiagents/finetuning/blob/main/Pompt_Prefix_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q peft transformers accelerate bitsandbytes --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00


In [ ]:
# !pip install -U bitsandbytes

In [2]:
model_name = "tiiuae/falcon-rw-1b"

In [3]:
import os, getpass

In [6]:
os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

In [8]:
from transformers import Trainer, TrainingArguments

In [9]:
from peft import get_peft_model
from peft import TaskType

In [10]:
# these are meant for either of PromptTuning or PrefixTuning
from peft import PromptTuningConfig, PrefixTuningConfig

In [11]:
# This is meant for LoRA only
from peft import LoraConfig

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [13]:
# device_map is on auto mode meaning its automatically set the device mode for which ever is available
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

**for LoRA based Fine Tuning**

In [13]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [14]:
peft_model3 = get_peft_model(model, config3)

In [15]:
peft_model3.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,416,220,672 || trainable%: 0.1111


In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
trainer = Trainer(
    model=peft_model3,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()


**Prompt Tuning**

In [ ]:
config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init="TEXT",
    prompt_tuning_init_text="Summarize the following:",
    num_virtual_tokens=8,
    tokenizer_name_or_path=model_name,
)

In [ ]:
peft_model = get_peft_model(model, config)
peft_model.print_trainable_parameters()

trainable params: 16,384 || all params: 1,311,641,600 || trainable%: 0.0012


**Prefix Tuning**

In [ ]:
config2 = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=8,
    prefix_projection=True,
)

In [ ]:
peft_model2 = get_peft_model(model, config2)
peft_model2.print_trainable_parameters()

trainable params: 205,637,632 || all params: 1,517,262,848 || trainable%: 13.5532


**LoRA Tuning**

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)
peft_model3.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,313,198,080 || trainable%: 0.1198


**Implementation of LoRA on Abirate dataset**

In [15]:
!pip install datasets --q

In [16]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset

In [17]:
# Example: load tiny dataset
dataset = load_dataset("Abirate/english_quotes")

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [18]:
dt = dataset["train"].select(range(100))

In [19]:
dt.features

{'quote': Value('string'),
 'author': Value('string'),
 'tags': List(Value('string'))}

In [20]:
def tokenize(example):
    return tokenizer(example["quote"], truncation=True, padding="max_length", max_length=128)


In [21]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Add a new pad token
    model.resize_token_embeddings(len(tokenizer)) # Resize model embeddings to include the new token
if tokenizer.pad_token is None or tokenizer.pad_token_id == -1:
    tokenizer.pad_token = tokenizer.eos_token # Fallback if no dedicated [PAD] was effectively set

In [22]:
dt.map(tokenize)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 100
})

In [23]:
tokenized = dt.map(tokenize)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [24]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [25]:
peft_model3 = get_peft_model(model, config3)

In [26]:
training_args = TrainingArguments(
    output_dir="./falcon_peft_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-4,
    logging_steps=10,
    fp16=True,
    save_strategy="no",
    report_to=[],
)


In [27]:
trainer = Trainer(
    model=peft_model3,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [28]:
trainer.train()

Step,Training Loss
10,1.073834
20,0.981655
30,0.925688
40,0.793120
50,0.704330
60,0.578943
70,0.583828


TrainOutput(global_step=75, training_loss=0.8041953404744466, metrics={'train_runtime': 23.1199, 'train_samples_per_second': 12.976, 'train_steps_per_second': 3.244, 'total_flos': 278824432435200.0, 'train_loss': 0.8041953404744466, 'epoch': 3.0})

In [29]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is NOT available. Please enable a GPU runtime in Colab.")
    print("Go to Runtime > Change runtime type, then select 'T4 GPU' or another available GPU as the hardware accelerator.")

GPU is available!
Device name: Tesla T4


In [31]:
peft_model3.save_pretrained("falcon-peft-tuned")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


In [43]:
# Load the base model first
from peft import PeftModel

model_name = "tiiuae/falcon-rw-1b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Ensure the tokenizer and model embeddings match the training setup
# Re-add the special pad token and resize embeddings, as was done before training
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Add a new pad token
    base_model.resize_token_embeddings(len(tokenizer)) # Resize model embeddings to include the new token
# Fallback if no dedicated [PAD] was effectively set during tokenization setup
if tokenizer.pad_token is None or tokenizer.pad_token_id == -1:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [1]:
# Load the PEFT model from the saved directory
peft_model_path = "./falcon-peft-tuned"
loaded_peft_model = PeftModel.from_pretrained(base_model, peft_model_path)

In [2]:

# Optional: Merge the LoRA weights into the base model for easier inference
merged_model = loaded_peft_model.merge_and_unload()

print("Model loaded and (optionally) merged successfully!")

Use `merged_model` (or `loaded_peft_model` if didn't merged) for inference, just like with any other Hugging Face model.

In [32]:
def build_prompt(review: str) -> str:
    return f"""### Instruction:
Classify the sentiment of the review.

### Input:
{review}

### Response:
"""

In [33]:
import torch

In [34]:
review = "The product quality was excellent but delivery was late."

prompt = build_prompt(review)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.0
    )

text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
prediction = text.split("Sentiment:")[-1].strip()

print("Prediction:", prediction)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Prediction: ### Instruction:
Classify the sentiment of the review.

### Input:
The product quality was excellent but delivery was late.

### Response:
We are so sorry to
